# 21 — Search+RAG Integration & Self-Consistency
Combines the beam search best prompt template with the RAG pipeline and runs a 4-way comparison:
- Baseline (zero-shot, no RAG)
- Search-only (beam search template, no RAG)
- RAG-only (context injection, base prompt)
- Search+RAG (beam search template + RAG context)

Also demonstrates self-consistency (5 samples, majority vote) with RAG enabled.
Results saved to `data/results/search_rag_comparison.json`.

In [1]:
import sys, os, json, time
from pathlib import Path
from collections import Counter, defaultdict
from dotenv import load_dotenv

# Resolve repo root reliably whether run via papermill or jupyter
def _find_repo_root():
    p = Path(os.path.abspath("."))
    for _ in range(6):
        if (p / "prompt-search").exists() and (p / "prompt-search" / "data").exists():
            return p / "prompt-search"
        if (p / "data" / "domain" / "courses.faiss").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find repo root")
REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env")

from retrieval.retriever import CourseRetriever
from retrieval.context_builder import ContextBuilder
from llm.claude_client import ClaudeClient
from evaluation.metrics import EvaluationMetrics
from prompts.template import PromptTemplate
from prompts.mutations import PromptMutator
from search.beam_search import BeamSearchPromptOptimizer

INDEX_DIR   = REPO_ROOT / "data" / "domain"
RESULTS_DIR = REPO_ROOT / "data" / "results"
TEST_CASES  = REPO_ROOT / "data" / "domain" / "test_cases.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

retriever = CourseRetriever(index_dir=str(INDEX_DIR))
builder   = ContextBuilder(retriever, top_k=5)
client    = ClaudeClient(model="claude-haiku-4-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

raw = json.loads(TEST_CASES.read_text())
test_cases = (raw["cases"] if isinstance(raw, dict) else raw)[:120]
print(f"Loaded {len(test_cases)} test cases.")

/Users/jaime/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaime/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 120 test cases.


## Step 1: Run Beam Search to Find Best Prompt Template

In [2]:
val_set = [
    {"question": tc["question"], "answer": tc["expected_answer"]}
    for tc in test_cases[:30]
]

base_template = PromptTemplate(
    system_role="a WSU academic advisor",
    task_description="Answer the student's question about WSU courses, prerequisites, or degree requirements. Be concise.",
)

optimizer = BeamSearchPromptOptimizer(
    beam_width=3,
    max_iterations=5,
    llm_client=client,
    patience=2,
)

print("Running beam search on 30-case validation set...")
t0 = time.time()
best_template = optimizer.search(base_template, val_set)
print(f"Done in {time.time()-t0:.1f}s")
print(f"Best mutation path: {best_template.mutation_path()}")
for entry in optimizer.history:
    print(f"  iter {entry['iteration']}: best_acc={entry['best_accuracy']:.3f} path={entry['best_path']}")

Running beam search on 30-case validation set...
Done in 1357.2s
Best mutation path: base
  iter 1: best_acc=0.000 path=add_cot
  iter 2: best_acc=0.000 path=add_cot -> add_cot


## Step 2: Define 4 Evaluation Modes

In [3]:
def run_mode(cases, mode, template=None):
    predictions, ground_truth = [], []
    for tc in cases:
        question = tc["question"]
        expected = tc["expected_answer"]

        if mode == "baseline":
            prompt = f"You are a WSU academic advisor.\n\nQuestion: {question}"
        elif mode == "search_only":
            prompt = template.render(question)
        elif mode == "rag_only":
            prompt, _ = builder.build(question)
        elif mode == "search_rag":
            rag_prompt, _ = builder.build(question)
            search_prefix = template.render("").replace("\n\nProblem: ", "")
            prompt = search_prefix + "\n\n" + rag_prompt

        resp = client.generate(prompt, temperature=0.0, max_tokens=400)
        predictions.append(resp)
        ground_truth.append(expected)

    acc = EvaluationMetrics.accuracy(predictions, ground_truth)
    return {"mode": mode, "accuracy": round(acc, 4), "predictions": predictions, "ground_truth": ground_truth}

print("Running 4-way comparison on all 120 cases...")
results = {}
for mode in ["baseline", "search_only", "rag_only", "search_rag"]:
    print(f"  {mode}...", end="", flush=True)
    t0 = time.time()
    results[mode] = run_mode(test_cases, mode, template=best_template)
    print(f" {results[mode]['accuracy']:.2%} ({time.time()-t0:.1f}s)")

Running 4-way comparison on all 120 cases...
  baseline... 30.00% (375.6s)
  search_only... 52.50% (269.7s)
  rag_only... 77.50% (194.0s)
  search_rag... 72.50% (198.4s)


## Step 3: 4-Way Comparison Table

In [4]:
cat_map = defaultdict(list)
for i, tc in enumerate(test_cases):
    cat_map[tc.get("category", "general")].append(i)

modes = ["baseline", "search_only", "rag_only", "search_rag"]
print(f"{'Category':<30} {'Baseline':>9} {'Search':>9} {'RAG':>9} {'Search+RAG':>11}")
print("-" * 72)

category_rows = []
for cat, idxs in sorted(cat_map.items()):
    accs = {}
    for mode in modes:
        preds = [results[mode]["predictions"][i] for i in idxs]
        gt    = [results[mode]["ground_truth"][i] for i in idxs]
        accs[mode] = EvaluationMetrics.accuracy(preds, gt)
    print(f"{cat:<30} {accs['baseline']:>9.2%} {accs['search_only']:>9.2%} {accs['rag_only']:>9.2%} {accs['search_rag']:>11.2%}")
    category_rows.append({"category": cat, **{m: round(accs[m], 4) for m in modes}})

print("-" * 72)
overall = {m: round(results[m]["accuracy"], 4) for m in modes}
print(f"{'OVERALL':<30} {overall['baseline']:>9.2%} {overall['search_only']:>9.2%} {overall['rag_only']:>9.2%} {overall['search_rag']:>11.2%}")

Category                        Baseline    Search       RAG  Search+RAG
------------------------------------------------------------------------
credit_calculations               50.00%    65.00%    75.00%      75.00%
degree_progress                   25.00%    30.00%    60.00%      65.00%
prerequisite_chain_discovery      32.00%    44.00%    80.00%      68.00%
prerequisite_validation           16.67%    63.33%    93.33%      80.00%
schedule_feasibility              26.67%    46.67%    60.00%      66.67%
ucore_planning                    40.00%    70.00%    90.00%      80.00%
------------------------------------------------------------------------
OVERALL                           30.00%    52.50%    77.50%      72.50%


## Step 4: Self-Consistency with RAG (5 samples, majority vote)

In [5]:
SC_SAMPLES = 5
SC_SUBSET  = test_cases[:20]  # run on 20 cases to limit API cost

def majority_vote(answers):
    return Counter(answers).most_common(1)[0][0]

sc_predictions, sc_ground_truth = [], []
print(f"Running self-consistency ({SC_SAMPLES} samples) on {len(SC_SUBSET)} cases...")
for tc in SC_SUBSET:
    prompt, _ = builder.build(tc["question"])
    samples = [client.generate(prompt, temperature=0.7, max_tokens=400) for _ in range(SC_SAMPLES)]
    sc_predictions.append(majority_vote(samples))
    sc_ground_truth.append(tc["expected_answer"])

sc_acc = EvaluationMetrics.accuracy(sc_predictions, sc_ground_truth)
rag_acc_20 = EvaluationMetrics.accuracy(
    [results["rag_only"]["predictions"][i] for i in range(len(SC_SUBSET))],
    sc_ground_truth
)
print(f"RAG only (20 cases):          {rag_acc_20:.2%}")
print(f"RAG + self-consistency (5x):  {sc_acc:.2%}")
print(f"Self-consistency delta:        {sc_acc - rag_acc_20:+.2%}")

Running self-consistency (5 samples) on 20 cases...
RAG only (20 cases):          90.00%
RAG + self-consistency (5x):  90.00%
Self-consistency delta:        +0.00%


## Step 5: Save Results

In [6]:
output = {
    "best_mutation_path": best_template.mutation_path(),
    "beam_search_history": optimizer.history,
    "overall": overall,
    "by_category": category_rows,
    "self_consistency": {
        "samples": SC_SAMPLES,
        "subset_size": len(SC_SUBSET),
        "rag_only_acc": round(rag_acc_20, 4),
        "sc_rag_acc": round(sc_acc, 4),
        "delta": round(sc_acc - rag_acc_20, 4),
    },
}
out_path = RESULTS_DIR / "search_rag_comparison.json"
out_path.write_text(json.dumps(output, indent=2))
print(f"Results saved to {out_path}")

Results saved to /Volumes/asus/VC-n8n-440/prompt-search/data/results/search_rag_comparison.json
